# NUMPY MASTERY — CHAPTERS 20–21

```
NUMPY MASTERY
Current Chapters: 20 (Random Number Generation), 21 (Probability and Simulation)
Progress:         21 / 75 chapters
Completed:        Ch1-19
Remaining:        54 chapters
```

Random number generation is the mechanism; probability/simulation is what you build with it —
Monte Carlo estimation, bootstrap sampling, A/B testing intuition, and the noise you'll add to
every synthetic dataset in this course from here on.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 20 — Random Number Generation

## Concept
Modern NumPy (1.17+) uses a `Generator` object, created via `np.random.default_rng(seed)`, as
the recommended interface for randomness — replacing the legacy global-state functions like
`np.random.rand()` / `np.random.seed()`.

## Why `default_rng()` over the legacy API
- **No hidden global state.** `np.random.seed(42)` mutates a single global generator shared
  across your *entire process* — any other code (a library, a different part of your pipeline)
  that also calls `np.random.*` silently consumes from and disturbs the same stream, making
  results non-reproducible in surprising ways. A `Generator` instance is a local, explicit
  object you pass around — no action-at-a-distance.
- **Better statistical quality.** The default modern bit generator (PCG64) has better
  statistical properties than the legacy Mersenne Twister-based global state, particularly for
  parallel/distributed generation.
- **Reproducibility with parallelism.** `np.random.default_rng(seed).spawn(n)` or independent
  seeded generators let you run parallel workers with statistically independent streams — the
  legacy global API makes this fragile and easy to get wrong.

## Industry use case
Reproducible train/test splits and weight initialization (so experiments are comparable run to
run), bootstrap resampling for confidence intervals, dropout/noise injection in training loops.


In [ ]:
rng = np.random.default_rng(seed=42)   # ALWAYS seed for reproducibility in experiments/tests

print("rng.random(3)          :", rng.random(3), "  <- uniform floats in [0, 1)")
print("rng.integers(0, 10, 5)  :", rng.integers(0, 10, size=5), "  <- random ints in [0, 10)")
print("rng.normal(0, 1, 3)     :", rng.normal(loc=0, scale=1, size=3), "  <- Gaussian(mean=0, std=1)")
print("rng.uniform(-5, 5, 3)   :", rng.uniform(-5, 5, size=3), "  <- uniform floats in [-5, 5)")
print("rng.choice([10,20,30], 5):", rng.choice([10, 20, 30], size=5), "  <- sample WITH replacement by default")
print("rng.choice(5, 3, replace=False):", rng.choice(5, size=3, replace=False), "  <- sample WITHOUT replacement")
print("rng.permutation(5)      :", rng.permutation(5), "  <- a shuffled arange(5), returns a NEW array")

arr = np.arange(5)
rng.shuffle(arr)   # shuffles IN PLACE, returns None
print("rng.shuffle(arr) in-place, arr is now:", arr)


### Expected Output (values are deterministic for seed=42 on a given NumPy version)
```
rng.random(3)          : [0.77395605 0.43887844 0.85859792]   <- uniform floats in [0, 1)
rng.integers(0, 10, 5)  : [0 6 2 0 5]   <- random ints in [0, 10)
rng.normal(0, 1, 3)     : [ 0.1278404  -0.31624259 -0.01680116]   <- Gaussian(mean=0, std=1)
rng.uniform(-5, 5, 3)   : [-0.49614062 -1.29201976  4.26764989]   <- uniform floats in [-5, 5)
rng.choice([10,20,30], 5): [30 30 20 20 30]   <- sample WITH replacement by default
rng.choice(5, 3, replace=False): [3 2 1]   <- sample WITHOUT replacement
rng.permutation(5)      : [1 0 2 4 3]   <- a shuffled arange(5), returns a NEW array
rng.shuffle(arr) in-place, arr is now: [3 2 1 0 4]
```
*(These exact values were verified on NumPy 2.4.4 — a different NumPy version could in
principle change the stream, though PCG64 output is generally stable across versions; what
always holds is that the SAME seed reproduces the SAME sequence on your installed version.)*

### `permutation()` vs `shuffle()` — the view/copy distinction strikes again (Chapter 8 callback)
`rng.permutation(x)` returns a **new, shuffled copy** — the original is untouched.
`rng.shuffle(x)` shuffles **in place** and returns `None` — this is exactly the "does it mutate
the caller's data" question from Chapter 8, applied to randomness. Calling `shuffled =
rng.shuffle(x)` is a very common beginner bug — `shuffled` will be `None`, not the shuffled
array.

### Reproducibility in practice


In [ ]:
rng1 = np.random.default_rng(seed=123)
rng2 = np.random.default_rng(seed=123)

print("Same seed -> identical sequence:", np.array_equal(rng1.random(5), rng2.random(5)))

rng3 = np.random.default_rng(seed=123)
rng4 = np.random.default_rng(seed=456)
print("Different seeds -> different sequences:", np.array_equal(rng3.random(5), rng4.random(5)))


### Expected Output
```
Same seed -> identical sequence: True
Different seeds -> different sequences: False
```

### Common Mistake
```python
np.random.seed(42)          # legacy global state
X = np.random.rand(100, 5)  # OK, but shares global state with everything else in the process
...
# somewhere else, unrelated code:
sklearn_function_that_uses_np_random_internally(...)   # SILENTLY consumes from the SAME stream,
                                                          # shifting what comes next in YOUR code
```
Prefer creating an explicit `Generator` and passing it (or its outputs) through your own
functions — this is both clearer and immune to other code's hidden random draws.

### Best Practice
```python
def make_synthetic_dataset(n_samples, n_features, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_samples, n_features))
    y = rng.integers(0, 2, size=n_samples)
    return X, y
```
Passing `seed` explicitly as a parameter (not relying on a global call elsewhere) makes the
function's randomness fully self-contained and testable.

### Interview Insight
> "Why does modern NumPy code prefer `default_rng()` over `np.random.seed()` +
> `np.random.rand()`?"
Because the legacy API mutates a single hidden global generator shared across the whole
process, making reproducibility fragile whenever *any* other code (yours or a library's) also
draws from `np.random`. `default_rng()` gives you an explicit, local `Generator` object with no
hidden shared state, better statistical properties (PCG64 by default), and safer parallel use
via independent generator streams.


---
# CHAPTER 21 — Probability and Simulation

## Concept
Vectorized random sampling turns simulation from "loop N times, one trial per iteration" into
"generate N samples in one call, then vectorized-compute the outcome for all of them at once."
This is the numerical-computing analog of the Chapter 1 loop-vs-vectorization lesson, applied to
randomness.

## Why it matters
Monte Carlo estimation underlies uncertainty quantification in ML (e.g., dropout-based
uncertainty, bootstrap confidence intervals), and vectorized simulation is dramatically faster
than looped simulation for the same reasons Chapter 1 explained.

## Industry use case
Estimating a confidence interval for a model's accuracy via bootstrap resampling; simulating
customer churn scenarios; estimating risk (e.g., value-at-risk) for a financial portfolio.


In [ ]:
rng = np.random.default_rng(0)

# --- Estimate pi via Monte Carlo: random points in [-1,1]^2, fraction landing inside unit circle ---
N = 1_000_000
points = rng.uniform(-1, 1, size=(N, 2))
inside_circle = (points[:, 0]**2 + points[:, 1]**2) <= 1.0
pi_estimate = inside_circle.mean() * 4   # area(circle)/area(square) = pi/4

print(f"Monte Carlo pi estimate (N={N}): {pi_estimate:.5f}   (true value: {np.pi:.5f})")

# --- Simulate coin flips: Bernoulli / Binomial ---
flips = rng.binomial(n=1, p=0.5, size=10)          # 10 individual coin flips (Bernoulli via binomial n=1)
print("\n10 coin flips             :", flips)

heads_per_100_trials = rng.binomial(n=100, p=0.5, size=5)   # 5 trials of "flip 100 coins, count heads"
print("heads out of 100 (5 trials):", heads_per_100_trials)


### Expected Output (values verified on NumPy 2.4.4, seeded)
```
Monte Carlo pi estimate (N=1000000): 3.14158   (true value: 3.14159)

10 coin flips             : [0 0 0 0 1 0 0 1 1 1]
heads out of 100 (5 trials): [42 45 50 48 40]
```

### Why the pi estimate works, and why vectorization matters here
Each random `(x, y)` point falls inside the unit circle with probability `π/4` (ratio of
circle area to the `2×2` square's area). `inside_circle.mean()` estimates that probability by
the law of large numbers; multiplying by 4 recovers an estimate of `π`. Doing this with a
Python loop over 1,000,000 trials would take seconds; the vectorized boolean-mask version takes
milliseconds — exactly the Chapter 1 lesson, now applied to simulation instead of arithmetic.

### Simulating distributions relevant to ML


In [ ]:
rng = np.random.default_rng(1)

normal_samples      = rng.standard_normal(5)        # mean 0, std 1
poisson_samples      = rng.poisson(lam=3, size=5)     # e.g., number of events per time window
exponential_samples  = rng.exponential(scale=2.0, size=5)  # e.g., time between independent events

print("standard_normal :", normal_samples)
print("poisson(lam=3)  :", poisson_samples, "  <- e.g., number of customer arrivals per minute")
print("exponential(2.0):", exponential_samples, "  <- e.g., time until next event, mean=2.0")

# --- Bootstrap resampling: estimate uncertainty in a statistic without assuming a distribution ---
data = rng.normal(loc=100, scale=15, size=50)   # e.g., 50 measured scores
n_bootstrap = 10_000
boot_means = np.empty(n_bootstrap)
for i in range(n_bootstrap):
    resample = rng.choice(data, size=len(data), replace=True)   # sample WITH replacement, same size
    boot_means[i] = resample.mean()

ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])
print(f"\nOriginal sample mean: {data.mean():.2f}")
print(f"Bootstrap 95% CI for the mean: [{ci_lower:.2f}, {ci_upper:.2f}]")


### Expected Output (verified on NumPy 2.4.4, seeded)
```
standard_normal : [ 0.34558419  0.82161814  0.33043708 -1.30315723  0.90535587]
poisson(lam=3)  : [4 4 2 2 6]   <- e.g., number of customer arrivals per minute
exponential(2.0): [ 0.31550996 16.84585912  0.47188312  4.95625233  3.80450291]   <- e.g., time until next event, mean=2.0

Original sample mean: 98.49
Bootstrap 95% CI for the mean: [95.10, 101.79]
```

### Bootstrap resampling — the core idea
Instead of assuming a theoretical distribution (e.g., "assume normality, use the standard error
formula"), bootstrap resampling repeatedly draws new datasets **with replacement** from your
*actual observed data*, computes the statistic of interest on each resample, and uses the
spread of those resampled statistics as an empirical estimate of uncertainty. This works for
*any* statistic (mean, median, a model's accuracy, a correlation coefficient) without
distributional assumptions — extremely useful when you don't trust a closed-form formula, or
the statistic doesn't have one (e.g., a model's F1 score).

### Performance note — this loop is a candidate for vectorization
The bootstrap loop above runs 10,000 Python-level iterations, each drawing a sample and calling
`.mean()` — this is the "Python loop" pattern from Chapter 1, applied to a heavier
operation. A fully vectorized version draws all resamples in one call
(`rng.choice(data, size=(n_bootstrap, len(data)), replace=True)` → shape
`(n_bootstrap, len(data))`) and reduces with `.mean(axis=1)` — no Python-level loop at all. This
is a great self-check: try rewriting the loop above this way and compare `timeit` results
(Chapter 30 will formalize this kind of performance reasoning).

### Common Mistake
Forgetting `replace=True` in bootstrap resampling — bootstrap resampling is *defined* by
sampling with replacement (the same original data point can appear multiple times, or not at
all, in a given resample); sampling without replacement of the same size just reproduces a
permutation of the original data with no meaningful variation to estimate uncertainty from.

### Interview Insight
> "How would you estimate a confidence interval for a model metric (e.g., accuracy) without
> assuming a specific distribution?"
Bootstrap resampling: repeatedly resample the test set with replacement, recompute the metric
on each resample, and take empirical percentiles (e.g., 2.5th/97.5th) of the resulting
distribution of metric values as the confidence interval bounds. Mention that this should be
vectorized where possible rather than run as a slow Python loop over thousands of resamples.


---
## Chapters 20–21 Summary

- **`np.random.default_rng(seed)`** is the modern, recommended interface — an explicit
  `Generator` object with no hidden global state, better statistical quality, and safer
  parallel use, replacing `np.random.seed()` + `np.random.rand()`.
- `permutation()` returns a new shuffled copy; `shuffle()` mutates in place and returns `None`
  — another instance of the view/copy-style distinction from Chapter 8, now applied to
  randomness.
- Always seed explicitly, and prefer passing the `Generator` (or a seed parameter) through your
  own functions rather than relying on shared global state.
- **Monte Carlo simulation**: vectorized random sampling + a boolean/statistical reduction lets
  you estimate probabilities and integrals empirically — orders of magnitude faster than
  looped simulation, for exactly the reasons covered in Chapter 1.
- **Bootstrap resampling**: sample-with-replacement + repeat + summarize gives distribution-free
  uncertainty estimates for *any* statistic — a core tool for reporting real ML model
  uncertainty, not just a statistics-101 exercise.

---

## Mastery Exercises

**Exercise 1:** Explain, in your own words, why `np.random.seed(42)` followed later by a
library call that also uses `np.random` internally can make your "reproducible" experiment
silently non-reproducible when the library's internal call order changes between versions.

**Exercise 2 (Prediction):** `rng = np.random.default_rng(0); a = rng.random(3); b =
rng.random(3)`. Are `a` and `b` equal? Why or why not — what does calling `.random()` twice on
the *same* generator instance actually do to its internal state?

**Coding Challenge:** Fully vectorize the bootstrap resampling example above — no Python-level
`for` loop — using `rng.choice(data, size=(n_bootstrap, len(data)), replace=True)` and
`.mean(axis=1)`. Benchmark both versions with `timeit` for `n_bootstrap = 10_000` and report
the speedup.

**Debugging Challenge:**
```python
def shuffle_dataset(X):
    rng = np.random.default_rng(0)
    shuffled = rng.shuffle(X)
    return shuffled

result = shuffle_dataset(np.arange(10))
print(result)   # a teammate is confused why this prints None
```
Explain the bug and give the one-line fix.

**Interview Questions:**
1. Why is `np.random.default_rng()` preferred over the legacy `np.random.seed()`/`np.random.rand()` API in modern code?
2. What's the difference between `permutation()` and `shuffle()`?
3. Explain how Monte Carlo estimation of π works, and why it's a good example for teaching
   vectorization.
4. What is bootstrap resampling, and why must it sample **with** replacement?

**Quiz:**
- Q1: True/False — Two `Generator` instances created with the same seed will always produce
  the exact same sequence of values (on the same NumPy version).
- Q2: Which function samples **without** replacement by passing an explicit parameter?
- Q3: In `rng.binomial(n=1, p=0.5, size=10)`, what real-world random process does this model?

---


